In [67]:
# ----------------------------------------------------------------
# 1. CORE IMPORTS
# ----------------------------------------------------------------
import os
import io
import gc
import math
import random
import time
import json
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
import torchvision.transforms as T
from torchvision.utils import make_grid

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, Image as IPImage

from datasets import load_dataset  # HuggingFace datasets

warnings.filterwarnings("ignore")

In [13]:
# ----------------------------------------------------------------
# 2. SEED / FULL REPRODUCIBILITY
# ----------------------------------------------------------------
SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

In [14]:
# ----------------------------------------------------------------
# 3. DEVICE-AGNOSTIC SETUP
# ----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] using: {device}")

def to_device(batch, device=device):
    """Recursively move tensors/dicts/lists/tuples to device."""
    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=True)
    if isinstance(batch, dict):
        return {k: to_device(v, device) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(v, device) for v in batch)
    return batch

[device] using: cuda


In [15]:
# ----------------------------------------------------------------
# 4. STANDARD DATA TRANSFORMS
# ----------------------------------------------------------------
IMG_MEAN = (0.4914, 0.4822, 0.4465)   # CIFAR-10 defaults; override per-paper if needed
IMG_STD  = (0.2470, 0.2435, 0.2616)

train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

In [24]:
# ----------------------------------------------------------------
# 5. STANDARD DATA TRANSFORMS (MNIST)
# ----------------------------------------------------------------
IMG_MEAN = (0.1307,)   # MNIST defaults; single channel (grayscale)
IMG_STD  = (0.3081,)

train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(IMG_MEAN, IMG_STD),
])

In [29]:
# ----------------------------------------------------------------
# 6. DATASET WRAPPER (HuggingFace -> PyTorch Dataset) — DATASET-AGNOSTIC
# ----------------------------------------------------------------
class HFImageDataset(Dataset):
    """
    Wraps a HuggingFace image dataset split into a torch-style Dataset.
    Expects each item to expose a PIL image under `image_key` and a
    label under `label_key` (label optional for unsupervised papers).

    `mode` controls the PIL color conversion:
      - "RGB" for 3-channel datasets (CIFAR-10, etc.)
      - "L"   for grayscale datasets (MNIST, Fashion-MNIST, etc.)
    """
    def __init__(self, hf_split, transform=None,
                 image_key: str = "img", label_key: Optional[str] = "label",
                 mode: str = "RGB"):
        self.data = hf_split
        self.transform = transform
        self.image_key = image_key
        self.label_key = label_key
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        img = item[self.image_key]
        if img.mode != self.mode:
            img = img.convert(self.mode)
        if self.transform:
            img = self.transform(img)
        if self.label_key is not None and self.label_key in item:
            return img, int(item[self.label_key])
        return img


# Per-dataset HF schema config: hub name, image/label keys, color mode
DATASET_REGISTRY: Dict[str, Dict[str, Any]] = {
    "cifar10": {"hub_name": "uoft-cs/cifar10", "image_key": "img",   "label_key": "label", "mode": "RGB"},
    "mnist":   {"hub_name": "ylecun/mnist",   "image_key": "image", "label_key": "label", "mode": "L"},
}


def load_image_subsets(dataset_name: str = "cifar10",
                        train_n: int = 2000, test_n: int = 400,
                        seed: int = SEED,
                        train_transform=train_transform,
                        eval_transform=eval_transform):
    """
    Standardized dataset loader: HF hub, small train/test subsets,
    matches the series' fixed replication settings. Works for any
    dataset registered in DATASET_REGISTRY (currently cifar10, mnist).
    """
    if dataset_name not in DATASET_REGISTRY:
        raise ValueError(
            f"Unknown dataset '{dataset_name}'. "
            f"Add it to DATASET_REGISTRY first. Known: {list(DATASET_REGISTRY)}"
        )
    cfg = DATASET_REGISTRY[dataset_name]

    ds = load_dataset(cfg["hub_name"])
    train_ds = ds["train"].shuffle(seed=seed).select(range(train_n))
    test_ds = ds["test"].shuffle(seed=seed).select(range(test_n))

    train_set = HFImageDataset(train_ds, transform=train_transform,
                                image_key=cfg["image_key"],
                                label_key=cfg["label_key"],
                                mode=cfg["mode"])
    test_set = HFImageDataset(test_ds, transform=eval_transform,
                               image_key=cfg["image_key"],
                               label_key=cfg["label_key"],
                               mode=cfg["mode"])
    return train_set, test_set


# Backward-compatible alias for CIFAR-10, keeps existing notebooks working unchanged
def load_cifar10_subsets(train_n: int = 2000, test_n: int = 400, seed: int = SEED):
    return load_image_subsets("cifar10", train_n, test_n, seed)

# Backward-compatible alias for MNIST, mirrors the CIFAR-10 one
def load_mnist_subsets(train_n: int = 2000, test_n: int = 400, seed: int = SEED):
    return load_image_subsets("mnist", train_n, test_n, seed)

In [17]:
# ----------------------------------------------------------------
# 7. DATA CLEANING / SANITY UTILITIES
# ----------------------------------------------------------------
def sanity_check_dataset(dataset: Dataset, n: int = 3) -> None:
    """Pull a few samples, check shape/dtype/range, catch schema issues early."""
    for i in range(min(n, len(dataset))):
        sample = dataset[i]
        img = sample[0] if isinstance(sample, tuple) else sample
        assert isinstance(img, torch.Tensor), f"Sample {i} is not a Tensor"
        assert img.dtype == torch.float32, f"Sample {i} dtype {img.dtype} != float32"
        assert not torch.isnan(img).any(), f"Sample {i} contains NaNs"
    print(f"[sanity] {n} samples checked — shapes/dtype/NaNs OK")


def normalize_labels(dataset, label_map: Dict[Any, int]):
    """Generic label remapper — use for AG News / text-label papers."""
    return [label_map[l] for l in dataset]

In [43]:
# ----------------------------------------------------------------
# 8. DATALOADER FACTORY (Colab-safe: num_workers=0)
# ----------------------------------------------------------------
def make_loaders(train_set, test_set, batch_size: int = 64,
                  num_workers: int = 0):
    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, generator=g, drop_last=True,
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False,
        num_workers=num_workers,
    )
    return train_loader, test_loader

In [68]:
# ----------------------------------------------------------------
# 9. CPU FORWARD-PASS SANITY CHECK (run BEFORE moving model to GPU)
# ----------------------------------------------------------------
def cpu_forward_check(model: nn.Module, sample_batch) -> None:
    """
    Runs one forward pass on CPU with a small batch to catch shape/dtype
    bugs before the (slower-to-debug) GPU move. Call this right after
    model instantiation, before `model.to(device)`.
    """
    model_cpu = model.to("cpu")
    model_cpu.eval()
    with torch.no_grad():
        x = sample_batch[0] if isinstance(sample_batch, (tuple, list)) else sample_batch
        out = model_cpu(x)
    print(f"[cpu-check] input {tuple(x.shape)} -> output "
          f"{tuple(out.shape) if isinstance(out, torch.Tensor) else type(out)}")
    del model_cpu
    gc.collect()

In [20]:
# ----------------------------------------------------------------
# 10. PARAM COUNTING / MISC MODEL UTILS
# ----------------------------------------------------------------
def count_params(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [21]:
# ----------------------------------------------------------------
# 11. STANDARD 8-PANEL DASHBOARD (white bg, black text, inline via BytesIO)
# ----------------------------------------------------------------
def render_dashboard(panel_fns: List[callable], title: str = "",
                      figsize: Tuple[int, int] = (16, 9)) -> None:
    """
    panel_fns: list of exactly 8 callables, each taking a matplotlib
    Axes and drawing one panel (loss curve, samples grid, metric, etc).
    """
    assert len(panel_fns) == 8, "Dashboard expects exactly 8 panel functions"

    fig = plt.figure(figsize=figsize, facecolor="white")
    gs = gridspec.GridSpec(2, 4, figure=fig)
    fig.suptitle(title, color="black", fontsize=14, fontweight="bold")

    for i, fn in enumerate(panel_fns):
        ax = fig.add_subplot(gs[i // 4, i % 4])
        ax.set_facecolor("white")
        fn(ax)
        ax.tick_params(colors="black")
        for spine in ax.spines.values():
            spine.set_color("black")

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", facecolor="white", dpi=120)
    plt.close(fig)
    buf.seek(0)
    display(IPImage(data=buf.read()))

In [23]:
# ----------------------------------------------------------------
# 12. TIMER / TRAINING UTILITIES
# ----------------------------------------------------------------
class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *args):
        self.elapsed = time.time() - self.t0


@dataclass
class RunConfig:
    epochs: int = 5
    batch_size: int = 64
    lr: float = 1e-3
    seed: int = SEED
    device: str = str(device)